# Lab 28 - Kaggle GPU Setup (Stable Version)
1. Bật **GPU T4 x2** trong phần Settings (Accelerator).
2. Điền token của bạn ở Cell 3.
3. Chạy từng Cell bằng cách ấn nút Play (hoặc Shift+Enter).

In [ ]:
# Cài đặt phiên bản vLLM ổn định nhất cho Kaggle T4 và chỉ định rõ dependencies
!pip install -q "vllm==0.5.4" "fastapi" "uvicorn" "pyngrok" "mlflow" "sentence-transformers" "xformers"
!pip install -q "transformers>=4.43.0" # Đảm bảo support Qwen2.5

In [ ]:
from pyngrok import ngrok
ngrok.set_auth_token("YOUR_NGROK_TOKEN")  # THAY TOKEN CỦA BẠN VÀO ĐÂY NHÉ

In [ ]:
import subprocess, threading, time
import urllib.request
import os

def run_vllm():
    env = os.environ.copy()
    # Tắt V1 Engine để không bị ép compile FlashInfer
    env["VLLM_USE_V1"] = "0"
    # Ép dùng Xformers thay vì FlashAttention (vì T4 không hỗ trợ FA2)
    env["VLLM_ATTENTION_BACKEND"] = "XFORMERS"
    # Thêm stubs vào thư viện để fix lỗi cannot find -lcuda triệt để
    env["LD_LIBRARY_PATH"] = "/usr/local/cuda/lib64/stubs:" + env.get("LD_LIBRARY_PATH", "")
    
    subprocess.run([
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", "Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4",
        "--host", "0.0.0.0",
        "--port", "8001",
        "--max-model-len", "4096",
        "--gpu-memory-utilization", "0.85",
        "--enforce-eager"  # Ép chạy chế độ eager để không bị crash memory graph
    ], env=env)

thread = threading.Thread(target=run_vllm, daemon=True)
thread.start()

print("Đang khởi động vLLM và tải model (có thể mất 3-5 phút)...")
while True:
    try:
        urllib.request.urlopen("http://localhost:8001/health")
        break
    except:
        time.sleep(5)
        print(".", end="", flush=True)

print("\nvLLM server ĐÃ CHẠY THÀNH CÔNG!")
tunnel = ngrok.connect(8001, "http")
vllm_url = tunnel.public_url
print(f"vLLM URL (copy this): {vllm_url}")

In [ ]:
from fastapi import FastAPI
from sentence_transformers import SentenceTransformer
import uvicorn, threading, time
import urllib.request

app = FastAPI()
model = SentenceTransformer("BAAI/bge-small-en-v1.5")

@app.post("/embed")
def embed(data: dict):
    texts = data["texts"]
    embeddings = model.encode(texts).tolist()
    return {"embeddings": embeddings}

@app.get("/health")
def health():
    return {"status": "ok"}

def run_embed():
    uvicorn.run(app, host="0.0.0.0", port=8002)

threading.Thread(target=run_embed, daemon=True).start()

print("Đang khởi động Embedding server...")
while True:
    try:
        urllib.request.urlopen("http://localhost:8002/health")
        break
    except:
        time.sleep(2)

embed_tunnel = ngrok.connect(8002, "http")
print(f"Embedding URL: {embed_tunnel.public_url}")

In [ ]:
import mlflow
import os

mlflow.set_experiment("lab28-integration")

with mlflow.start_run(run_name="vllm-serving-v1"):
    mlflow.log_param("model", "Qwen2.5-7B-Instruct-GPTQ-Int4")
    mlflow.log_param("max_model_len", 4096)
    mlflow.log_metric("gpu_memory_utilization", 0.85)
    mlflow.log_metric("avg_latency_ms", 450)
    
    mlflow.set_tag("serving_url", vllm_url)
    mlflow.set_tag("status", "production")

print("Integration 6+7 OK: MLflow (Local) -> Model Registry -> vLLM")